## 1. Imports and Notebook Configuration
Import required libraries and set up notebook-wide display and plotting options.

## Outline
1. Imports and Notebook Configuration
2. Historical Price Data Acquisition
3. Heikin-Ashi Candle Construction
4. Trend & Signal Logic (Bull/Bear Detection)
5. Vectorized Backtest Engine (Entries/Exits)
6. Performance & Risk Metrics Computation
7. Stop-Loss / Take-Profit & Position Sizing
8. Parameter Grid / Random Search Optimization
9. Walk-Forward (Out-of-Sample) Evaluation
10. Visualization: Standard vs Heikin-Ashi with Signals
11. Signal Export (CSV / JSON) & Run Logging
12. Lightweight Caching & Incremental Updates
13. Unit Test Helpers (Functions to Be Tested)
14. Optional CLI Entrypoint for Terminal Execution

In [ ]:
import pandas as pd
import numpy as np
import yfinance as yf
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objs as go
import os
from datetime import datetime, timedelta

# Set display and plotting options
pd.set_option('display.max_columns', 20)
sns.set(style='darkgrid')
plt.rcParams['figure.figsize'] = (14, 6)

# Global constants
default_ticker = 'AAPL'
default_interval = '1d'
default_start = '2020-01-01'
default_end = datetime.today().strftime('%Y-%m-%d')
risk_free_rate = 0.03

def get_data(ticker=default_ticker, start=default_start, end=default_end, interval=default_interval, cache_dir='./cache'):
    os.makedirs(cache_dir, exist_ok=True)
    cache_file = f"{cache_dir}/{ticker}_{start}_{end}_{interval}.csv"
    if os.path.exists(cache_file):
        df = pd.read_csv(cache_file, index_col=0, parse_dates=True)
    else:
        df = yf.download(ticker, start=start, end=end, interval=interval, auto_adjust=True)
        df.to_csv(cache_file)
    df = df.tz_localize(None)
    df = df.dropna()
    return df

df = get_data()
df.head()

def compute_heikin_ashi(df):
    ha_df = df.copy()
    ha_close = (df['Open'] + df['High'] + df['Low'] + df['Close']) / 4
    ha_open = ha_close.copy()
    ha_open.iloc[0] = (df['Open'].iloc[0] + df['Close'].iloc[0]) / 2
    for i in range(1, len(df)):
        ha_open.iloc[i] = (ha_open.iloc[i-1] + ha_close.iloc[i-1]) / 2
    ha_high = pd.concat([df['High'], ha_open, ha_close], axis=1).max(axis=1)
    ha_low = pd.concat([df['Low'], ha_open, ha_close], axis=1).min(axis=1)
    ha_df['HA_Open'] = ha_open
    ha_df['HA_Close'] = ha_close
    ha_df['HA_High'] = ha_high
    ha_df['HA_Low'] = ha_low
    return ha_df

ha_df = compute_heikin_ashi(df)
ha_df[['Open','High','Low','Close','HA_Open','HA_High','HA_Low','HA_Close']].head()

def detect_trend(df, ema_len=6):
    ha_close = df['HA_Close']
    ha_open = df['HA_Open']
    trend = np.where(ha_close > ha_open, 1, -1)
    if ema_len:
        trend = pd.Series(trend, index=df.index).ewm(span=ema_len, adjust=False).mean().apply(lambda x: 1 if x > 0 else -1)
    return trend

def generate_signals(df, ema_len=6):
    trend = detect_trend(df, ema_len)
    signals = trend.shift(1).fillna(0)
    return signals

ha_df['Trend'] = detect_trend(ha_df)
ha_df['Signal'] = generate_signals(ha_df)
ha_df[['HA_Open','HA_Close','Trend','Signal']].tail()


# Vectorized Backtest Engine

def run_backtest(df, signals, initial_equity=10000, slippage=0.0005, commission=0.0002):
    df = df.copy()
    df['Position'] = signals
    df['Position'] = df['Position'].replace(0, method='ffill').fillna(0)
    df['Market_Return'] = df['Close'].pct_change()
    df['Strategy_Return'] = df['Market_Return'] * df['Position']
    df['Strategy_Return'] -= (abs(df['Position'].diff()) * (slippage + commission)).fillna(0)
    df['Equity'] = initial_equity * (1 + df['Strategy_Return']).cumprod()
    return df

bt_df = run_backtest(ha_df, ha_df['Signal'])
bt_df[['Close','Position','Equity']].tail()


# Performance & Risk Metrics

def compute_metrics(df, risk_free_rate=0.03):
    returns = df['Strategy_Return'].dropna()
    total_return = df['Equity'].iloc[-1] / df['Equity'].iloc[0] - 1
    cagr = (df['Equity'].iloc[-1] / df['Equity'].iloc[0]) ** (252/len(df)) - 1
    max_dd = ((df['Equity'].cummax() - df['Equity']) / df['Equity'].cummax()).max()
    sharpe = (returns.mean() - risk_free_rate/252) / (returns.std() + 1e-9) * np.sqrt(252)
    sortino = (returns.mean() - risk_free_rate/252) / (returns[returns<0].std() + 1e-9) * np.sqrt(252)
    hit_ratio = (returns > 0).mean()
    avg_win = returns[returns > 0].mean()
    avg_loss = returns[returns < 0].mean()
    exposure = (df['Position'] != 0).mean()
    metrics = {
        'Total Return': total_return,
        'CAGR': cagr,
        'Max Drawdown': max_dd,
        'Sharpe': sharpe,
        'Sortino': sortino,
        'Hit Ratio': hit_ratio,
        'Avg Win': avg_win,
        'Avg Loss': avg_loss,
        'Exposure': exposure
    }
    return pd.DataFrame([metrics])

metrics_df = compute_metrics(bt_df)
metrics_df.T

# ATR-based Stop-Loss, Take-Profit, and Position Sizing

def add_atr(df, period=14):
    high_low = df['High'] - df['Low']
    high_close = np.abs(df['High'] - df['Close'].shift())
    low_close = np.abs(df['Low'] - df['Close'].shift())
    ranges = pd.concat([high_low, high_close, low_close], axis=1)
    true_range = ranges.max(axis=1)
    atr = true_range.rolling(window=period).mean()
    df['ATR'] = atr
    return df

def position_size(equity, risk_per_trade, atr, stop_mult):
    return (equity * risk_per_trade) / (atr * stop_mult)

ha_df = add_atr(ha_df)
# Example: position sizing for 1% risk, 2x ATR stop
ha_df['Position_Size'] = position_size(10000, 0.01, ha_df['ATR'], 2)
ha_df[['ATR','Position_Size']].tail()

# Parameter Grid / Random Search Optimization
from itertools import product
import random

def optimize_params(df, param_grid, n_iter=10, random_search=True):
    keys, values = zip(*param_grid.items())
    if random_search:
        combos = [dict(zip(keys, [random.choice(v) for v in values])) for _ in range(n_iter)]
    else:
        combos = [dict(zip(keys, v)) for v in product(*values)]
    results = []
    for params in combos:
        ha_df = compute_heikin_ashi(df)
        ha_df['Trend'] = detect_trend(ha_df, ema_len=params['ema_len'])
        ha_df['Signal'] = generate_signals(ha_df, ema_len=params['ema_len'])
        bt_df = run_backtest(ha_df, ha_df['Signal'])
        metrics = compute_metrics(bt_df).iloc[0].to_dict()
        metrics.update(params)
        results.append(metrics)
    return pd.DataFrame(results)

param_grid = {'ema_len': [4, 6, 8, 10]}
opt_results = optimize_params(df, param_grid, n_iter=5, random_search=True)
opt_results.sort_values('Sharpe', ascending=False).head()

def walk_forward(df, param_grid, window=252, step=126):
    metrics_list = []
    for start in range(0, len(df) - window, step):
        train = df.iloc[start:start+window]
        test = df.iloc[start+window:start+window+step]
        opt_results = optimize_params(train, param_grid, n_iter=3, random_search=True)
        best_params = opt_results.sort_values('Sharpe', ascending=False).iloc[0]
        ha_test = compute_heikin_ashi(test)
        ha_test['Trend'] = detect_trend(ha_test, ema_len=int(best_params['ema_len']))
        ha_test['Signal'] = generate_signals(ha_test, ema_len=int(best_params['ema_len']))
        bt_test = run_backtest(ha_test, ha_test['Signal'])
        metrics = compute_metrics(bt_test).iloc[0].to_dict()
        metrics['start'] = test.index[0]
        metrics['end'] = test.index[-1]
        metrics_list.append(metrics)
    return pd.DataFrame(metrics_list)

wf_metrics = walk_forward(df, param_grid, window=252, step=126)
wf_metrics[['start','end','Sharpe','CAGR','Max Drawdown']]

# Visualization: Standard vs Heikin-Ashi with Signals
import mplfinance as mpf

def plot_ha_signals(df, bt_df):
    fig, axes = plt.subplots(3, 1, figsize=(14, 12), sharex=True)
    axes[0].plot(df.index, df['Close'], label='Close', color='black')
    axes[0].set_title('Standard OHLC Close')
    axes[1].plot(df.index, df['HA_Close'], label='HA Close', color='blue')
    axes[1].set_title('Heikin-Ashi Close')
    axes[2].plot(bt_df.index, bt_df['Equity'], label='Equity Curve', color='green')
    axes[2].set_title('Equity Curve')
    for ax in axes:
        ax.legend()
    plt.tight_layout()
    plt.show()

plot_ha_signals(ha_df, bt_df)


# Signal Export & Run Logging
import json
from pathlib import Path

def export_artifacts(df, metrics, out_dir='./artifacts'):
    Path(out_dir).mkdir(exist_ok=True)
    ts = datetime.now().strftime('%Y%m%d_%H%M%S')
    df.to_csv(f'{out_dir}/signals_{ts}.csv')
    metrics.to_json(f'{out_dir}/metrics_{ts}.json', orient='records')
    with open(f'{out_dir}/run_log.txt', 'a') as f:
        f.write(f'{ts}: Run completed. Sharpe={metrics["Sharpe"].values[0]:.2f}\n')

export_artifacts(bt_df, metrics_df)

# Lightweight Caching & Incremental Updates

def cache_key(ticker, start, end, interval):
    return f"{ticker}_{start}_{end}_{interval}"

def update_heikin_ashi_cache(df, cache_dir='./cache'):
    key = cache_key(default_ticker, default_start, default_end, default_interval)
    cache_file = f"{cache_dir}/ha_{key}.csv"
    if os.path.exists(cache_file):
        cached = pd.read_csv(cache_file, index_col=0, parse_dates=True)
        if len(df) > len(cached):
            new_rows = df.iloc[len(cached):]
            new_ha = compute_heikin_ashi(new_rows)
            updated = pd.concat([cached, new_ha])
            updated.to_csv(cache_file)
            return updated
        return cached
    else:
        ha_df = compute_heikin_ashi(df)
        ha_df.to_csv(cache_file)
        return ha_df

ha_df = update_heikin_ashi_cache(df)
